In [0]:
-- =====================================================
-- BUSINESS QUESTION:
-- Which assessment types and modules have the highest failure rates?
--
-- Grain:
-- One row per score per assessment
--
-- Sources:
-- dim_assessment
-- fact_assessment
-- =====================================================

CREATE OR REPLACE TABLE
    `ftw-week-07`.`04-analytics`.assessment_failure_rates
AS

WITH assessment_failure_rates AS (
    SELECT
        da.code_module,
        da.assessment_type,
        fa.score,
        CASE
            -- Scores below 40 are considered failures
            WHEN fa.score < 40 THEN 1
            -- Conditionally include NULLs based on parameter
            WHEN fa.score IS NULL AND :include_nulls_as_failures = 'Yes' THEN 1
            ELSE 0
        END AS is_failure

    FROM `ftw-week-07`.`03-mart`.fact_assessment fa
    LEFT JOIN `ftw-week-07`.`03-mart`.dim_assessment da
        ON fa.assessment_key = da.assessment_key
)

SELECT
    code_module,
    assessment_type,
    COUNT(*) AS total_assessments,
    SUM(CASE WHEN score IS NULL THEN 1 ELSE 0 END) AS null_scores,
    ROUND(SUM(CASE WHEN score IS NULL THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS null_pct,
    SUM(is_failure) AS total_failures,
    ROUND(SUM(is_failure) * 100 / COUNT(*), 2) AS failure_rate_pct,
    ROUND(AVG(score), 2) AS average_score
FROM assessment_failure_rates
GROUP BY code_module, assessment_type
ORDER BY failure_rate_pct DESC;

code_module,assessment_type,total_assessments,null_scores,null_pct,total_failures,failure_rate_pct,average_score
CCC,Exam,1915,0,0.0,232,12.11,68.91
DDD,Exam,3044,0,0.0,335,11.01,63.47
CCC,CMA,9766,0,0.0,928,9.5,73.14
DDD,TMA,22568,49,0.22,2053,9.1,70.66
DDD,CMA,5252,0,0.0,448,8.53,71.51
CCC,TMA,7259,11,0.15,611,8.42,74.57
BBB,TMA,27074,53,0.2,1358,5.02,70.01
FFF,TMA,24823,46,0.19,1036,4.17,75.14
AAA,TMA,3149,3,0.1,93,2.95,69.03
GGG,TMA,5660,4,0.07,153,2.7,68.47


Databricks visualization. Run in Databricks to view.